# 02 · App Gradio — Pruebas de la interfaz completa (SALVAGUARDA — venv Python 3.11)

Copia de seguridad de `app_gradio 4.ipynb`, adaptada para instalar todo dentro
de un entorno aislado de **Python 3.11** en vez del Python 3.13 que trae
Colab por defecto ahora mismo.

Motivo: varios paquetes que usa esta app (`deepfilternet`, `audiosr`,
`sgmse`, `clearvoice`...) fijan `numpy<2.0`, y numpy 1.x no publica wheels
para Python 3.13 — eso obligaba a compilar numpy (y a veces el propio
paquete) desde código fuente, tardando 15-20 minutos o colgándose
directamente. Python 3.11 sí tiene wheels para todo esto, igual que la
versión de Colab con la que se construyó originalmente esta notebook
(Python 3.12).

El notebook original (`app_gradio 4.ipynb`) se deja intacto como referencia;
esta es la versión "buena" a partir de ahora. No sustituye a las notebooks
de validación individual de cada modelo (01_DeepFilterNet..., etc.) — esas
siguen siendo la referencia para el bloque bibliográfico/empírico de la
memoria.

**IMPORTANTE:** este entorno de ejecución debe tener GPU asignada (Entorno
de ejecución → Cambiar tipo de entorno de ejecución → GPU T4). Comprueba
con la celda siguiente antes de continuar.


In [2]:
import torch
print(torch.__version__)
print(torch.version.cuda)


2.11.0+cu128
12.8


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = '/content/drive/MyDrive/TFG_restauracion_audio'  # <- ajustar si se clonó en otra ubicación
os.environ['PROJECT_ROOT'] = PROJECT_ROOT
os.environ['HF_HOME'] = f'{PROJECT_ROOT}/cache'
os.environ['HF_HUB_CACHE'] = f'{PROJECT_ROOT}/cache'
os.environ['PATH'] = f"/content/env311/bin:{os.environ['PATH']}"


## Crear el entorno aislado de Python 3.11

Se instala Python 3.11 como paquete del sistema (no compila nada, es
binario) y se crea un venv aparte. A partir de aquí, **todas** las
instalaciones de dependencias de los modelos van a `/content/env311/bin/pip`,
no a `!pip install` a secas — así quedan aisladas de los paquetes por
defecto de Colab (Python 3.13) y no hay conflicto de versiones de numpy
entre modelos.

Al ser un venv completamente aislado, no hereda nada de lo que Colab trae
preinstalado (ni siquiera PyTorch con soporte CUDA) — hay que instalarlo
todo explícitamente dentro, incluido PyTorch.


In [4]:
!apt-get update -qq
!apt-get install -y python3.11 python3.11-venv python3.11-dev -qq
!python3.11 --version


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libpython3.11-minimal:amd64.
(Reading database ... 122579 files and directories currently installed.)
Preparing to unpack .../0-libpython3.11-minimal_3.11.15-1+jammy1_amd64.deb ...
Unpacking libpython3.11-minimal:amd64 (3.11.15-1+jammy1) ...
Selecting previously unselected package python3.11-minimal.
Preparing to unpack .../1-python3.11-minimal_3.11.15-1+jammy1_amd64.deb ...
Unpacking python3.11-minimal (3.11.15-1+jammy1) ...
Selecting previously unselected package libpython3.11-stdlib:amd64.
Preparing to unpack .../2-libpython3.11-stdlib_3.11.15-1+jammy1_amd64.deb ...
Unpacking libpython3.11-stdlib:amd64 (3.11.15-1+jammy1) ...
Selecting previously unselected package libpython3.11:amd64.
Preparing to unpack .../3-libpython3.11_3.11.15-1+jammy1_amd64.deb ...
Unp

In [5]:
!python3.11 -m venv /content/env311
!/content/env311/bin/pip install --upgrade pip -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 14.5 MB/s eta 0:00:00


### PyTorch con soporte CUDA dentro del venv

**Ojo:** `pip install torch` a secas NO garantiza una build con CUDA — por
defecto puede darte la más reciente (p. ej. una compilada para CUDA 13.0),
que no funciona si el driver de la máquina no la soporta (nos pasó:
`2.14.0+cu130` con `CUDA disponible: False`). La forma fiable es clonar
exactamente la versión de PyTorch y de CUDA que ya usa Colab en el kernel
del notebook (comprobado en la primera celda de este notebook: `2.11.0` /
CUDA `12.8`), apuntando al índice de wheels correspondiente. Si la primera
celda de este notebook te dio una versión de torch o de CUDA distinta,
cambia los números de la línea de abajo para que coincidan.


In [6]:
!/content/env311/bin/pip install -q torch==2.11.0 torchaudio torchvision --index-url https://download.pytorch.org/whl/cu128


In [7]:
!/content/env311/bin/python -c "import torch; print('torch', torch.__version__, '| CUDA disponible:', torch.cuda.is_available())"


torch 2.11.0+cu128 | CUDA disponible: True


## Instalar dependencias comunes

Dos instalaciones separadas: una en el kernel del propio notebook (para
las celdas sueltas de utilidad, como la de generar audio de baja
resolución más abajo, que usan `files.upload()`/`files.download()` y por
tanto tienen que vivir en el kernel de Colab), y otra en el venv (para que
la app pueda usarlas al arrancar).

In [8]:
# Kernel del notebook (para las celdas de utilidad sueltas)
!pip install -q librosa soundfile scipy pesq pystoi speechmos onnxruntime matplotlib pandas


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


In [9]:
# Venv (para la app)
!/content/env311/bin/pip install -q librosa soundfile scipy pesq pystoi speechmos onnxruntime matplotlib pandas


## Dependencias de DeepFilterNet3 (denoising)

Igual que en `01_DeepFilterNet_3_6.ipynb`: hace falta el toolchain de Rust
para compilar `DeepFilterLib`, pero al no depender ya de compilar numpy
desde cero (Python 3.11 sí tiene wheel de `numpy<2.0`), la compilación de
Rust en sí es rápida — es un crate pequeño, no todo numpy.

Aquí usamos el paquete `deepfilternet` **oficial** (no el fork
`DeepFilterNet-py312`), porque este último exige `numpy>=2.0`, lo cual
choca con el `numpy<2.0` que necesitan AudioSR y sgmse en el resto de esta
misma notebook. Con Python 3.11 el oficial ya no tiene el problema de
wheel que nos obligó a usar el fork en Python 3.13.


In [10]:
import os

!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
os.environ['PATH'] = f"{os.environ['HOME']}/.cargo/bin:{os.environ['PATH']}"
!rustc --version


info: downloading installer
info: profile set to default
info: default host tuple is x86_64-unknown-linux-gnu
info: syncing channel updates for stable-x86_64-unknown-linux-gnu
info: latest update on 2026-09-03 for version 1.98.1 (48a229cea 2026-09-01)
info: downloading 6 components
      rustfmt installed                        2.37 MiB                         info: default toolchain set to stable-x86_64-unknown-linux-gnu

  stable-x86_64-unknown-linux-gnu installed - rustc 1.98.1 (48a229cea 2026-09-01)


Rust is installed now. Great!

To get started you may need to restart your current shell.
This would reload your PATH environment variable to include
Cargo's bin directory ($HOME/.cargo/bin).

To configure your current shell, you need to source the
corresponding env file under $HOME/.cargo.

Consider running the right command for your shell (note the leading DOT):
. "$HOME/.cargo/env" # For sh/ash/dash/pdksh/bash
cargo:rerun-if-env-changed=CC_x86_64-unknown-linux-gnu
CC_x86_64-unknown

In [11]:
!/content/env311/bin/pip install -q deepfilternet


In [12]:
# No hace falta reiniciar el entorno de ejecución: al vivir en un venv
# aparte, esto no toca los paquetes del kernel del notebook.
!/content/env311/bin/python -c "import numpy; print('numpy en el venv (debe ser 1.26.x):', numpy.__version__)"


numpy en el venv (debe ser 1.26.x): 1.26.4


## Dependencias de MP-SENet (dereverberation — caso de estudio, ver limitación en 4.4.2)

El paquete de inferencia `MPSENet` (envoltorio de JacobLinCool sobre los
checkpoints originales del autor, publicados en Hugging Face) es puro
Python — no hace falta compilar nada con Rust.


In [13]:
!/content/env311/bin/pip install -q MPSENet


## Dependencias de VoiceFixer v2 (de-clipping)

El paquete `voicefixer` declara en su `setup.py` versiones propias de
`librosa`/`matplotlib` que pueden entrar en conflicto con lo que ya tienen
instalado otros modelos de esta app (mismo tipo de problema que ya vimos
con AudioSR). Se instala con `--no-deps` y se añaden a mano solo las
dependencias que de verdad faltan, para no arriesgarnos a que reinstale
versiones antiguas de librosa/matplotlib por debajo.

El aviso de que falta `streamlit` es inofensivo: es solo para la demo de
línea de comandos propia de VoiceFixer, que no usamos aquí.


In [14]:
!/content/env311/bin/pip install -q --no-deps voicefixer
!/content/env311/bin/pip install -q torchlibrosa progressbar GitPython pyyaml


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
voicefixer 0.1.3 requires streamlit>=1.12.0, which is not installed.


## Dependencias de HTDemucs v4 (separación de fuentes)

Se usa el paquete `demucs-infer` (fork de solo inferencia de Demucs,
compatible con PyTorch 2.x) en vez del paquete `demucs` original, porque
este último fija `torchaudio<2.2` y entraría en conflicto con la versión
moderna de torchaudio que ya necesitan DeepFilterNet3 y el resto de
modelos de esta notebook.


In [15]:
!/content/env311/bin/pip install -q demucs-infer


## Dependencias de AudioSR (super-resolución/BWE)

`audiosr==0.0.7` declara `numpy<=1.23.5`, `librosa==0.9.2` y
`transformers==4.30.2`. Se instala con `--no-deps` y se añaden a mano solo
las dependencias transitorias que de verdad hacen falta, dejando que
numpy/librosa se queden en la versión que ya tiene el resto de la app
(1.26.x) — es el mismo patrón ya validado en `05_AudioSR.ipynb`. Si pip
avisa de conflicto de versiones aquí, no lo canceles: en la práctica no
ha impedido que el resto de la instalación siga; lo confirmamos de verdad
cuando probemos audio real por este modelo en el barrido de comprobación.


In [16]:
!/content/env311/bin/pip install -q audiosr==0.0.7 --no-deps
!/content/env311/bin/pip install -q unidecode phonemizer ftfy torchlibrosa
!/content/env311/bin/pip install -q transformers==4.30.2 --no-deps
!/content/env311/bin/pip install -q tokenizers safetensors regex
!/content/env311/bin/pip install -q --no-deps --force-reinstall "tokenizers>=0.11.1,!=0.11.3,<0.14"
!/content/env311/bin/pip install -q --no-deps timm chardet


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
audiosr 0.0.7 requires chardet, which is not installed.
audiosr 0.0.7 requires gradio, which is not installed.
audiosr 0.0.7 requires timm, which is not installed.
audiosr 0.0.7 requires transformers==4.30.2, which is not installed.
audiosr 0.0.7 requires librosa==0.9.2, but you have librosa 0.11.0 which is incompatible.
audiosr 0.0.7 requires numpy<=1.23.5, but you have numpy 1.26.4 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 4.30.2 requires huggingface-hub<1.0,>=0.14.1, but you have huggingface-hub 1.30.0 which is incompatible.
transformers 4.30.2 requires tokenizers!=0.11.3,<0.14,>=0.11.1, but you have tokenizers 0.23.2 which is incompatible.


## Dependencias de ClearVoice/MossFormer2 (modelo combinado)

El paquete `clearvoice` descarga automáticamente los checkpoints de
MossFormer2_SE_48K y MossFormer2_SR_48K desde Hugging Face la primera vez
que se usan (de ahí que convenga tener el login de HF_TOKEN ya hecho antes
de llegar aquí). Si en algún momento se trabaja con formatos distintos de
`.wav`, `clearvoice` necesita FFmpeg — Colab ya lo trae instalado por
defecto, así que no hace falta instalarlo aparte.


In [17]:
!/content/env311/bin/pip install -q clearvoice


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
audiosr 0.0.7 requires gradio, which is not installed.
voicefixer 0.1.3 requires streamlit>=1.12.0, which is not installed.
audiosr 0.0.7 requires librosa==0.9.2, but you have librosa 0.10.2.post1 which is incompatible.
audiosr 0.0.7 requires numpy<=1.23.5, but you have numpy 1.26.4 which is incompatible.


## Instalar Gradio

In [18]:
# Gradio se instala normal (es la primera vez en este venv, no hace falta
# --force-reinstall) para que arrastre TODAS sus dependencias propias
# (fastapi, uvicorn, pydantic, etc.), no solo el paquete en si.
!/content/env311/bin/pip install -q gradio==6.20.0

# Gradio 6.20 arrastra numpy 2.x y "packaging" reciente como parte de su
# propia resolucion de dependencias, lo cual choca con deepfilternet y
# clearvoice (numpy<2.0) instalados antes. Antes se fijaban numpy y
# packaging en DOS "pip install --force-reinstall" SEPARADOS, y cada uno
# deshacia un poco lo que habia fijado el anterior (de ahi el vaiven de
# versiones -- 2.4.6, luego 1.26.4, packaging quedandose en 26.3-- que
# veias en los logs). Aqui se fijan los dos A LA VEZ, en una sola
# resolucion, con --no-deps (numpy y packaging no tienen dependencias
# propias, asi que no hay riesgo de dejar nada a medias):
!/content/env311/bin/pip install -q --no-deps --force-reinstall numpy==1.26.4 "packaging>=23.0,<24.0"


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
audiosr 0.0.7 requires librosa==0.9.2, but you have librosa 0.10.2.post1 which is incompatible.
audiosr 0.0.7 requires numpy<=1.23.5, but you have numpy 1.26.4 which is incompatible.


### Comprobación final de versiones y de que los módulos importan de verdad

Importante: **todo** este chequeo tiene que pasar por
`/content/env311/bin/python`, en una única llamada — si se ejecuta como
código Python suelto en una celda normal, se comprueban los paquetes del
kernel del notebook (Python 3.13), no los del venv, y los números que salen
no significan nada sobre si la app va a funcionar (esto es justo lo que
pasó la vez anterior).

Los avisos de pip tipo `ERROR: pip's dependency resolver does not
currently take into account...` que has visto durante toda la instalación
son en su mayoría ruido esperado, no fallos reales: `audiosr` y
`voicefixer` declaran en su propio `setup.py` dependencias (`chardet`,
`timm`, `transformers==4.30.2`, `streamlit`, un `gradio` propio) que solo
usa el código de sus demos de línea de comandos — código que esta app
nunca llama. Pip avisa igualmente porque cada `pip install` solo comprueba
compatibilidad contra lo que instala en ese mismo comando, no contra todo
lo que ya tenías instalado de antes. Lo que de verdad importa es si cada
módulo **importa** correctamente con las versiones finales que quedan —
eso es lo que comprueba la celda de abajo (además de imprimir versiones):
si algo aparece como `FALLO`, ahí sí hay un problema real que atender; si
todo dice `OK`, los avisos de pip de arriba eran solo ruido.


In [19]:
%%writefile /content/check_versiones.py

import huggingface_hub

def _compatibilizar_hf_hub_download(func_original):
    def wrapper(*args, **kwargs):
        if 'use_auth_token' in kwargs:
            kwargs.setdefault('token', kwargs.pop('use_auth_token'))
        return func_original(*args, **kwargs)
    return wrapper

huggingface_hub.hf_hub_download = _compatibilizar_hf_hub_download(huggingface_hub.hf_hub_download)
huggingface_hub.file_download.hf_hub_download = huggingface_hub.hf_hub_download
if hasattr(huggingface_hub, 'snapshot_download'):
    huggingface_hub.snapshot_download = _compatibilizar_hf_hub_download(huggingface_hub.snapshot_download)

    # Parche 4: torchaudio.load() en esta version exige el paquete "torchcodec"
# (no instalado). Lo evitamos leyendo el audio con soundfile directamente,
# manteniendo la misma interfaz (tensor, sample_rate) que espera el resto
# del codigo (audiosr, deepfilternet, etc.).
import torch
import torchaudio
import soundfile as sf

def _torchaudio_load_via_soundfile(filepath, *args, **kwargs):
    data, sr = sf.read(filepath, always_2d=True)
    waveform = torch.from_numpy(data.T).float()
    return waveform, sr

torchaudio.load = _torchaudio_load_via_soundfile

import sys, os, types

# Parche VoiceFixer: forzar backend headless de matplotlib
os.environ['MPLBACKEND'] = 'Agg'

# Parche DeepFilterNet: recrear el modulo torchaudio.backend retirado
import torchaudio
if not hasattr(torchaudio, 'backend'):
    backend_mod = types.ModuleType('torchaudio.backend')
    common_mod = types.ModuleType('torchaudio.backend.common')

    class AudioMetaData:
        def __init__(self, sample_rate=0, num_frames=0, num_channels=0,
                     bits_per_sample=0, encoding=""):
            self.sample_rate = sample_rate
            self.num_frames = num_frames
            self.num_channels = num_channels
            self.bits_per_sample = bits_per_sample
            self.encoding = encoding

    common_mod.AudioMetaData = AudioMetaData
    backend_mod.common = common_mod
    sys.modules['torchaudio.backend'] = backend_mod
    sys.modules['torchaudio.backend.common'] = common_mod
    torchaudio.backend = backend_mod

import importlib

print('--- Comprobacion funcional real (import de cada modelo) ---')
comprobaciones = {
    'DeepFilterNet3': lambda: __import__('df.enhance', fromlist=['enhance']),
    'MP-SENet': lambda: __import__('MPSENet'),
    'VoiceFixer': lambda: __import__('voicefixer'),
    'HTDemucs/demucs-infer': lambda: __import__('demucs_infer.api', fromlist=['Separator']),
    'AudioSR': lambda: __import__('audiosr'),
    'ClearVoice': lambda: __import__('clearvoice'),
    'SGMSE+': lambda: __import__('sgmse'),
    'Gradio': lambda: __import__('gradio'),
}
fallos = []
for nombre, importar in comprobaciones.items():
    try:
        importar()
        print(f'{nombre}: OK')
    except Exception as e:
        print(f'{nombre}: FALLO -> {type(e).__name__}: {e}')
        fallos.append(nombre)

print()
if fallos:
    print('ATENCION -- fallan de verdad:', ', '.join(fallos))
else:
    print('Todos los modulos importan bien.')


Writing /content/check_versiones.py


# Token de Hugging Face

In [20]:
try:
    from huggingface_hub import login
    from google.colab import userdata
    login(token=userdata.get('TOKEN_TFG_CLEARMOSS'))
except Exception:
    print("Sin token de HF: las descargas serán algo más lentas, pero funcionará igual.")


#Token de Ngrok

In [21]:
import sys
!{sys.executable} -m pip install -q pyngrok

In [ ]:
from pyngrok import ngrok
from google.colab import userdata

# Token propio de ngrok, guardado como secreto de Colab (icono de la llave
# en la barra lateral) bajo el nombre 'NGROK_AUTH_TOKEN'. Ver README para
# crear una cuenta gratuita de ngrok y obtener el token.
ngrok.set_auth_token(userdata.get('NGROK_AUTH_TOKEN'))


## Instalar nuevo modelo de dereverb funcional (SGMSE+)

Sustituye a MP-SENet como variante activa de la categoría (ver `dereverb.py`
nuevo). Ya no hace falta excluir torch/torchaudio/torchvision de la
instalación por miedo a romper el CUDA de Colab — en el venv los hemos
instalado nosotros explícitamente un poco más arriba, así que aquí basta
con el resto de dependencias propias de sgmse (sin el pin `numpy<2.0` de
su `requirements.txt`, ya cubierto por el numpy 1.26.4 fijado para
Gradio).

In [ ]:

!git clone https://github.com/sp-uhh/sgmse.git /content/sgmse
!/content/env311/bin/pip install -e /content/sgmse --no-deps -q

!/content/env311/bin/pip install -q gdown h5py ipympl librosa ninja pandas pesq pillow protobuf \
    pyarrow pyroomacoustics pystoi pytorch-lightning scipy sdeint \
    seaborn torch-ema torch-pesq torchinfo torchsde tqdm wandb

# gradio_app/pesos/ puede no existir todavía dentro del repo -- gdown no la crea sola.
!mkdir -p {PROJECT_ROOT}/gradio_app/pesos

!/content/env311/bin/pip install -q gdown
!/content/env311/bin/python -m gdown 1eiOy0VjHh9V9ZUFTxu1Pq2w19izl9ejD \
    -O {PROJECT_ROOT}/gradio_app/pesos/sgmse_wsj0_reverb.ckpt
!/content/env311/bin/pip install -q ninja

#Checkeo de los modelos

In [24]:
!/content/env311/bin/python /content/check_versiones.py


--- Comprobacion funcional real (import de cada modelo) ---
DeepFilterNet3: OK
MP-SENet: OK
Weights downloaded in: /root/.cache/voicefixer/synthesis_module/44100/model.ckpt-1490000_trimed.pt Size: 135613039
Weights downloaded in: /root/.cache/voicefixer/analysis_module/checkpoints/vf.ckpt Size: 489307071
VoiceFixer: OK
HTDemucs/demucs-infer: OK
/content/env311/lib/python3.11/site-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `hf_hub_download`. Downloads always resume whenever possible.
  warnings.warn(
/content/env311/lib/python3.11/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
AudioSR: OK
ClearVoice: OK
SGMSE+: OK
Gradio: OK

Todos los modulos importan bien.


## Lanzar la app (precarga de MossFormer2 + Gradio)

Cambio importante respecto al notebook original: antes se hacía
`import app` directamente en el kernel del notebook. Eso ya no funciona
aquí, porque el kernel del notebook sigue siendo Python 3.13 y no ve nada
de lo instalado en `/content/env311` — son dos instalaciones de Python
totalmente separadas. En su lugar, escribimos un script pequeño y lo
lanzamos con el intérprete del venv como proceso aparte.

`share=False` porque no se va a desplegar como Hugging Face Space ni se
va a repartir un enlace público — el acceso a la app en Colab se hace con
el proxy propio de Colab (celda de `proxyPort` más abajo), no con el
túnel de Gradio.

**Cambio de flujo respecto a antes:** el script se lanza en segundo plano
(`nohup ... &`) en vez de en una celda bloqueante. Así no hace falta
interrumpir la celda para poder ejecutar la siguiente — interrumpirla es
justo lo que antes mataba el servidor de Gradio (verás en el log
`Keyboard interruption... closing server`), y por eso ni el enlace local
ni el del proxy funcionaban después. El orden correcto ahora es: lanzar en
segundo plano → comprobar en el log que arrancó → coger el enlace del
proxy de Colab → abrirlo.


In [ ]:
%%writefile /content/launch_app_venv.py

import huggingface_hub

def _compatibilizar_hf_hub_download(func_original):
    def wrapper(*args, **kwargs):
        if 'use_auth_token' in kwargs:
            kwargs.setdefault('token', kwargs.pop('use_auth_token'))
        return func_original(*args, **kwargs)
    return wrapper

huggingface_hub.hf_hub_download = _compatibilizar_hf_hub_download(huggingface_hub.hf_hub_download)
huggingface_hub.file_download.hf_hub_download = huggingface_hub.hf_hub_download
if hasattr(huggingface_hub, 'snapshot_download'):
    huggingface_hub.snapshot_download = _compatibilizar_hf_hub_download(huggingface_hub.snapshot_download)

    # Parche 4: torchaudio.load() en esta version exige el paquete "torchcodec"
# (no instalado). Lo evitamos leyendo el audio con soundfile directamente,
# manteniendo la misma interfaz (tensor, sample_rate) que espera el resto
# del codigo (audiosr, deepfilternet, etc.).
import torch
import torchaudio
import soundfile as sf

#parche 5

_torch_load_original = torch.load
def _torch_load_compat(*args, **kwargs):
    kwargs['weights_only'] = False
    return _torch_load_original(*args, **kwargs)
torch.load = _torch_load_compat

def _torchaudio_load_via_soundfile(filepath, *args, **kwargs):
    data, sr = sf.read(filepath, always_2d=True)
    waveform = torch.from_numpy(data.T).float()
    return waveform, sr

torchaudio.load = _torchaudio_load_via_soundfile

import sys, os

PROJECT_ROOT = os.environ.get('PROJECT_ROOT', '/content/drive/MyDrive/TFG_restauracion_audio')
os.environ['HF_HOME'] = f'{PROJECT_ROOT}/cache'
os.environ['HF_HUB_CACHE'] = f'{PROJECT_ROOT}/cache'
sys.path.insert(0, f'{PROJECT_ROOT}/gradio_app')

import app

print("Precargando MossFormer2...")
app.mossformer2.model_manager.obtener_modelo(
    "combinado", "mossformer2_se", app.mossformer2._cargar_mossformer2_se
)
app.mossformer2.model_manager.obtener_modelo(
    "combinado", "mossformer2_sr", app.mossformer2._cargar_mossformer2_sr
)
print("MossFormer2 listo.")

app.demo.queue()
app.demo.launch(server_name="0.0.0.0", server_port=7860, share=False, debug=True, ssr_mode=False)


In [29]:
!rm -rf /content/checkpoints/MossFormer2_SR_48K

In [30]:
!pkill -9 -f launch_app_venv.py
!nohup /content/env311/bin/python -u /content/launch_app_venv.py > /content/gradio.log 2>&1 &
!sleep 25 && cat /content/gradio.log

Precargando MossFormer2...
downloading checkpoint for MossFormer2_SR_48K
Fetching 6 files:  83%|████████▎ | 5/6 [00:11<00:02,  2.23s/it]

In [27]:
!cat /content/gradio.log

Precargando MossFormer2...


In [31]:
public_url = ngrok.connect(7860, "http")
print(public_url)

NgrokTunnel: "https://booth-taps-humbly.ngrok-free.dev" -> "http://localhost:7860"


##Comprobaciones


In [29]:
!tail -f /content/gradio.log

Precargando MossFormer2...
downloading checkpoint for MossFormer2_SE_48K
Fetching 4 files: 100%|██████████| 4/4 [00:02<00:00,  1.73it/s]
downloading checkpoint for MossFormer2_SR_48K
Fetching 6 files: 100%|██████████| 6/6 [00:14<00:00,  2.40s/it]
MossFormer2 listo.
* Running on local URL:  http://0.0.0.0:7860
* To create a public link, set `share=True` in `launch()`.
Loading AudioSR: basic
Loading model on cuda
DiffusionWrapper has 258.20 M params.
Running DDIM Sampling with 50 timesteps
DDIM Sampler: 100%|██████████| 50/50 [00:12<00:00,  4.07it/s]
Running DDIM Sampling with 50 timesteps
DDIM Sampler: 100%|██████████| 50/50 [00:09<00:00,  5.12it/s]
Running DDIM Sampling with 50 timesteps
DDIM Sampler: 100%|██████████| 50/50 [00:10<00:00,  4.93it/s]
Audio guardado en: /tmp/salida_ia.wav
Audio guardado en: /tmp/salida_baseline.wav
^C


# Calcular las métricas
Un aviso importante antes de correrlo: si la app (launch_app_venv.py) sigue corriendo en segundo plano en esa misma sesión, párala primero — dos procesos intentando cargar modelos en la misma GPU (la app por un lado, el script de métricas por otro) te puede petar la VRAM del T4. Mata el proceso de la app antes de lanzar el script de métricas:

In [32]:
!pkill -9 -f launch_app_venv.py

Y luego ya corres las celdas de instalar pesq/pystoi y ejecutar calcular_metricas.py que te pasé.

In [33]:
!/content/env311/bin/python -m pip install -q pesq pystoi

In [35]:
!/content/env311/bin/python /content/drive/MyDrive/Proyecto_Audio/gradio_app/calcular_metricas.py

39 condiciones en el manifiesto.

[1/39] denoising frag1 snr_db=-5
2026-09-06 15:09:38 | INFO     | DF | Running on torch 2.11.0+cu128
2026-09-06 15:09:38 | INFO     | DF | Running on host 6eef2104f10d
fatal: not a git repository (or any of the parent directories): .git
2026-09-06 15:09:38 | INFO     | DF | Loading model settings of DeepFilterNet3
2026-09-06 15:09:38 | INFO     | DF | Using DeepFilterNet3 model at /root/.cache/DeepFilterNet/DeepFilterNet3
2026-09-06 15:09:38 | INFO     | DF | Initializing model `deepfilternet3`
2026-09-06 15:09:38 | INFO     | DF | Found checkpoint /root/.cache/DeepFilterNet/DeepFilterNet3/checkpoints/model_120.ckpt.best with epoch 120
2026-09-06 15:09:38 | INFO     | DF | Running on device cuda:0
2026-09-06 15:09:38 | INFO     | DF | Model loaded
2026-09-06 15:09:38 | WARNING  | DF | Audio sampling rate does not match model sampling rate (44100, 48000). Resampling...
/content/env311/lib/python3.11/site-packages/df/io.py:106: UserWarning: "sinc_interpo

#Recalcular metricas faltantes:

In [ ]:
!/content/env311/bin/python /content/drive/MyDrive/Proyecto_Audio/gradio_app/calcular_metricas1.py